# Serial chunked nonlinear registration

Use serial chunking when a whole-volume nonlinear optimization does not fit comfortably in memory. Affine estimation still happens once for the whole image; nonlinear warps are estimated and checkpointed per chunk.

In [ ]:
# EDIT THESE PATHS and parameters before running the notebook.
from pathlib import Path

FIXED_IMAGE = Path("path/to/fixed_reference.tif")
MOVING_IMAGE = Path("path/to/moving_round.tif")
OUTPUT_DIRECTORY = Path("path/to/registration_output")
ROUND_NUMBER = 2
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
CHUNK_SIZE_ZYX = (512, 512, 512)
OVERLAP_FRACTION = 0.15
LINEAR_ACCURACY = "low"
NONLINEAR_ACCURACY = "high"
ITK_THREADS = 1
BLAS_THREADS = 1

Pixel size is public XYZ spacing in micrometers. Chunk sizes follow the NumPy array order ZYX. Completed local warp records are reused after an interruption.

In [ ]:
from cyexm.registration import ChunkedImageRegistration
from cyexm.runtime import NativeThreadConfig

registration = ChunkedImageRegistration(
    fixed_image_path=FIXED_IMAGE,
    moving_image_path=MOVING_IMAGE,
    save_directory=OUTPUT_DIRECTORY,
    imaging_round=ROUND_NUMBER,
    pixel_size=PIXEL_SIZE_XYZ,
    chunk_size=CHUNK_SIZE_ZYX,
    overlap_fraction=OVERLAP_FRACTION,
    native_threads=NativeThreadConfig(ITK_THREADS, BLAS_THREADS),
    linear_accuracy=LINEAR_ACCURACY,
    nonlinear_accuracy=NONLINEAR_ACCURACY,
)

In [ ]:
registration.register()
REGISTERED_IMAGE = OUTPUT_DIRECTORY / f"nonlinear_registered_{ROUND_NUMBER}.tif"
print(f"Registration complete: {REGISTERED_IMAGE}")

The following optional view compares middle Z slices. It uses local NumPy, tifffile, and Matplotlib operations rather than a project-specific plotting helper.

In [ ]:
import matplotlib.pyplot as plt
import tifffile

fixed_array = tifffile.imread(FIXED_IMAGE)
moving_array = tifffile.imread(MOVING_IMAGE)
registered_array = tifffile.imread(REGISTERED_IMAGE)
z_index = registered_array.shape[0] // 2
figure, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, image, title in zip(
    axes,
    (fixed_array, moving_array, registered_array),
    ("Fixed", "Moving", "Registered"),
    strict=True,
):
    axis.imshow(image[z_index], cmap="gray")
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()